# 04 — Grad-CAM Explainability

**Kaggle:** same inputs as notebook 03 (dataset + output of 02b); Accelerator = GPU; Internet = On.

Theory: [`docs/06_gradcam.md`](../docs/06_gradcam.md)

In [ ]:
# --- environment setup: works on Kaggle, Colab and locally ---
import sys, os
REPO = "https://github.com/Mohd2Shaan/Deepfake-Detection-Project.git"
ON_KAGGLE = os.path.exists("/kaggle/input")
ON_COLAB = "google.colab" in sys.modules
if ON_KAGGLE or ON_COLAB:
    PROJECT = "/kaggle/working/project" if ON_KAGGLE else "/content/project"
    if os.path.exists(PROJECT):
        !git -C {PROJECT} pull -q
    else:
        !git clone -q {REPO} {PROJECT}
    %cd {PROJECT}
    EXTRA = "" if ON_KAGGLE else "kagglehub"
    !pip install -q grad-cam {EXTRA}
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("working dir:", os.getcwd())

In [ ]:
# --- dataset: Kaggle -> attached input is found automatically; Colab/local -> downloaded once (~4 GB) ---
from src import config
if ON_KAGGLE:
    for ckpt in config.import_kaggle_checkpoints():      # notebooks 03/04: models from notebook 02's output
        print("trained model:", ckpt)
if not (config.DATA_DIR / "train").exists():
    if ON_KAGGLE:
        raise RuntimeError("Dataset not found - attach '140k Real and Fake Faces' with Add Input")
    !python scripts/download_dataset.py --move
print("data dir:", config.DATA_DIR)

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from PIL import Image
from src import config
from src.model import load_checkpoint, get_target_layer
from src.gradcam import GradCAMExplainer, manual_gradcam, save_heatmap_grid, pick_correct_examples

device = config.get_device()
model, ckpt = load_checkpoint(config.CHECKPOINT_DIR / 'efficientnet_best.pth', device)
name = ckpt['model_name']
explainer = GradCAMExplainer(model, name, device)

## 1. Correctly classified real and fake faces
The misclassified grid needs the error list from notebook 03 — it is recomputed here if missing.

In [ ]:
if not list(config.MISCLASSIFIED_DIR.glob('*_most_confident_errors.csv')):
    !python -m src.evaluate --num-workers 4

In [ ]:
!python -m src.gradcam --checkpoint checkpoints/efficientnet_best.pth --n 12
from IPython.display import Image as Show, display
for p in sorted(config.HEATMAP_DIR.glob(f'{name}_*.png')):
    display(Show(str(p)))

## 2. Library Grad-CAM vs hand-written Grad-CAM
Both should produce (almost) the same map — shows we understand what the library computes.

In [ ]:
import numpy as np
path = pick_correct_examples(explainer, config.DATA_DIR, 1, 'fake')[0]
img = Image.open(path)
lib = explainer.explain(img, target_class=1)['cam']
x = explainer.transform(img.convert('RGB')).unsqueeze(0).to(device)
mine = manual_gradcam(model, get_target_layer(model, name), x, sign=1.0)
print('correlation:', np.corrcoef(lib.ravel(), mine.ravel())[0, 1])
fig, ax = plt.subplots(1, 3, figsize=(10, 3.5))
ax[0].imshow(img); ax[1].imshow(lib, cmap='jet'); ax[2].imshow(mine, cmap='jet')
for a, t in zip(ax, ['image', 'pytorch-grad-cam', 'manual']): a.set_title(t); a.axis('off')

## 3. ResNet-18 heatmaps for comparison

In [ ]:
!python -m src.gradcam --checkpoint checkpoints/resnet_best.pth --n 8

## Observations

_Where does the model look for fakes? (eyes, hairline, teeth, background edges?) Where does it look on misclassified images?_